# 03: Evaluation

In Notebook 02 we ran individual questions by hand. This notebook evaluates the agent
systematically: we upload a dataset subset to Langfuse, run the agent on every item, and
score each response with an LLM-as-judge grader using the official DeepSearchQA methodology.

## What You'll Learn

1. Uploading a DeepSearchQA subset to Langfuse as a persistent dataset
2. The LLM-as-judge grader: precision, recall, F1, and the four outcome categories
3. A single-sample evaluation walkthrough
4. Running the full experiment with `run_experiment`
5. Inspecting and interpreting item-level results

## Prerequisites

Complete Notebooks 01 and 02. You'll need all credentials in `.env`:
- `GOOGLE_API_KEY`
- `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY`
- `OPENAI_API_KEY` (for the LLM grader)

In [21]:
import json
import os
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.evaluation import run_experiment
from aieng.agent_evals.evaluation.graders.config import LLMRequestConfig
from aieng.agent_evals.knowledge_qa import BloombergFinancialNewsDataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.bloombergfinance_grader import (
    EvaluationOutcome,
    evaluate_bloomberg_groundtruth_async,
)
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from aieng.agent_evals.langfuse import upload_dataset_to_langfuse
from dotenv import load_dotenv
from IPython.display import HTML, display  # noqa: A004
from langfuse.experiment import Evaluation
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


# Set working directory to the repository root
if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)

console = Console(width=100)

DATASET_NAME = "Bloomberg-Financial"

Working directory: /home/coder/eval-agents


In [22]:
dataset = BloombergFinancialNewsDataset()
dataset.sample(20)

2026-04-19 21:35:42,756 INFO aieng.agent_evals.knowledge_qa.data.bloombergfinance: Loading golden eval data from /home/coder/eval-agents/aieng-eval-agents/aieng/agent_evals/knowledge_qa/data/golden_eval.json...
2026-04-19 21:35:42,762 INFO aieng.agent_evals.knowledge_qa.data.bloombergfinance: Loaded 30 golden eval examples


[BloombergNewsExample(input="Who was named Scotiabank's new Chief Risk Officer in August 2013?", expected_output='Hart, replacing Pitfield who retired', metadata=GoldenEvalMetadata(example_id=9, category='personnel', answer_type='Single Answer', bank='Scotiabank', question_type='single_article', source_date='2013-08-29', source_headline='Scotiabank Names Hart Chief Risk Officer as Pitfield Retires')),
 BloombergNewsExample(input="What was Royal Bank of Canada's Q4 2013 net income?", expected_output='This information is not available in the Bloomberg articles from late August 2013. Q4 2013 results had not been reported yet.', metadata=GoldenEvalMetadata(example_id=21, category='unanswerable', answer_type='Single Answer', bank='RBC', question_type='unanswerable', source_date='N/A', source_headline='N/A - Q4 results not yet reported')),
 BloombergNewsExample(input='Both BMO and Scotiabank in Q3 2013 reported record profits in which business segments?', expected_output='Both banks posted r

## 1. Uploading the Dataset to Langfuse

Langfuse stores our evaluation dataset so we can run multiple experiments against the same items
and compare results over time. Each dataset item has three fields:

- **`input`**: the question (sent to the agent)
- **`expected_output`**: the ground truth answer (given to the grader, never shown to the agent)
- **`metadata`**: `category`, `answer_type`, `example_id`

Items are deduplicated by a hash of their content, so running this cell again is safe.

In [24]:
# dataset = BloombergFinancialNewsDataset()
# # examples = dataset.get_by_category("Finance & Economics")[:1]
# examples = dataset.sample(20)
# console.print(f"Uploading [cyan]{len(examples)}[/cyan] examples to dataset '{DATASET_NAME}'...")

# # Write examples to a temporary JSONL file for the upload utility
# with tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False, encoding="utf-8") as f:
#     for ex in examples:
#         record = {
#             "input": ex.problem,
#             "expected_output": ex.answer,
#             "metadata": {
#                 "example_id": ex.example_id,
#                 "category": ex.problem_category,
#                 "answer_type": ex.answer_type,
#             },
#         }
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")
#     temp_path = f.name

temp_path = "/home/coder/eval-agents/aieng-eval-agents/aieng/agent_evals/knowledge_qa/data/golden_eval.json"
await upload_dataset_to_langfuse(dataset_path=temp_path, dataset_name=DATASET_NAME)
# os.unlink(temp_path)

# console.print(f"[green]✓[/green] Dataset '{DATASET_NAME}' ready in Langfuse")

2026-04-19 21:36:56,152 INFO aieng.agent_evals.langfuse: Loading dataset from '/home/coder/eval-agents/aieng-eval-agents/aieng/agent_evals/knowledge_qa/data/golden_eval.json'


Uploading Langfuse dataset 'Bloomberg-Financial' ━━━━━━━━━ 30/30 0:00:00 0:00:0200:0200:01


2026-04-19 21:36:58,591 INFO aieng.agent_evals.langfuse: Uploaded 30 items to dataset 'Bloomberg-Financial'


## 2. The Bloomberg Grader

The grader is an LLM-as-judge that evaluates answers using the official Bloomberg methodology
from Appendix A of the paper. It handles both answer types:

- **Single Answer**: checks whether the response contains the one expected value
- **Set Answer**: checks which items from the ground truth set appear in the response,
  and flags any extra items the agent included

### Metrics

Let **S** = predicted items, **G** = ground truth items:

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Precision** | \|S∩G\| / \|S\| | Of what the agent said, how much was correct |
| **Recall** | \|S∩G\| / \|G\| | Of the ground truth, how much did the agent find |
| **F1** | 2·P·R / (P+R) | Harmonic mean of precision and recall |

### Outcome Classification

| Outcome | Condition | Interpretation |
|---------|-----------|----------------|
| `fully_correct` | S = G | Perfect answer |
| `correct_with_extraneous` | G ⊆ S | All correct, but extra items included |
| `partially_correct` | S∩G ≠ ∅ | Some correct items found |
| `fully_incorrect` | S∩G = ∅ | No correct items |

### 2.1 Single-Sample Walkthrough

Before running at scale, let's walk through one example end-to-end: run the agent,
then grade its response with the LLM judge.

In [36]:
# Pick one example for the walkthrough
example = dataset[13]

console.print(
    Panel(
        f"[bold]ID:[/bold] {example.metadata.example_id}\n"
        f"[bold]Category:[/bold] {example.metadata.category}\n"
        f"[bold]Answer Type:[/bold] {example.metadata.answer_type}\n"
        f"[bold]Bank:[/bold] {example.metadata.bank}\n\n"
        f"[bold cyan]Question:[/bold cyan]\n{example.input}\n\n"
        f"[bold yellow]Ground Truth:[/bold yellow]\n{example.expected_output}",
        title="Golden Evaluation Example",
        border_style="blue",
    )
)

╭─────────────────────────────────── Golden Evaluation Example ────────────────────────────────────╮
│ ID: 14                                                                                           │
│ Category: comparative                                                                            │
│ Answer Type: Set Answer                                                                          │
│ Bank: all                                                                                        │
│                                                                                                  │
│ Question:                                                                                        │
│ Which of Canada's Big Five banks raised their dividends in Q3 2013 and by how much?              │
│                                                                                                  │
│ Ground Truth:                                                                                    │
│ Royal Bank of Canada raised its dividend 6.3 percent to 67 cents, Toronto-Dominion raised 4.9    │
│ percent to 85 cents, and Bank of Nova Scotia raised 3.3 percent to 62 cents                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [37]:
eval_agent = KnowledgeGroundedAgent(enable_planning=True)
eval_response = await run_with_display(eval_agent, example.input)

display_response(
    console,
    eval_response.text,
    title="Agent Answer",
    subtitle=f"Duration: {eval_response.total_duration_ms / 1000:.1f}s  |  Tools: {len(eval_response.tool_calls)}",
)

╭────────────────────────────────────────── Agent Answer ──────────────────────────────────────────╮
│ RBC, TD, and Scotiabank raised dividends in Q3 2013.                                             │
╰───────────────────────────────── Duration: 138.8s  |  Tools: 6 ──────────────────────────────────╯

╭───────────────────────────────────────────── Impact ─────────────────────────────────────────────╮
│ Neutral | Medium significance                                                                    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── Key Figures ───────────────────────────────────────────╮
│                                                                                                  │
│  • Royal Bank of Canada (RBC): Raised quarterly common share dividend by C$0.04 (6%) to C$0.67   │
│    per share.                                                                                    │
│  • Toronto-Dominion Bank (TD): Raised quarterly common share dividend by C$0.04 (4.9%) to C$0.85 │
│    per share.                                                                                    │
│  • Bank of Nova Scotia (Scotiabank): Raised quarterly dividend by C$0.02 to C$0.62 per share.    │
│  • Bank of Montreal (BMO): No dividend increase in Q3 2013.                                      │
│  • Canadian Imperial Bank of Commerce (CIBC): No dividend increase in Q3 2013 (a previous        │
│    increase was announced in Q2 2013).                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Analysis ────────────────────────────────────────────╮
│ In Q3 2013, three of Canada's Big Five banks — RBC, TD, and Scotiabank — announced increases to  │
│ their quarterly common share dividends, reflecting strong financial performance during the       │
│ period. BMO and CIBC maintained their dividends at prior quarter levels.                         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Sources ─────────────────────────────────────────────╮
│   1. RBC — 2013-08-29 — [RBC Q3 2013 Earnings News                                               │
│ Release](https://www.rbc.com/newsroom/news/article.html?article=123529)                          │
│   2. TD — 2013-08-29 — [TD Bank Group Reports Third Quarter 2013                                 │
│ Results](https://www.td.com/content/dam/tdcom/canada/about-td/pdf/quarterly-results/2013/2013-Q3 │
│ _Earnings_News_Release_F_EN.pdf)                                                                 │
│   3. CityNews — 2013-08-27 — [Scotiabank sees decline in Q3 net earnings                         │
│   raises quarterly                                                                               │
│ dividend](https://toronto.citynews.ca/2013/08/27/scotiabank-sees-decline-in-q3-net-earnings-rais │
│ es-quarterly-dividend/)                                                                          │
│   4. CBC News — 2013-08-27 — [Scotiabank hikes dividend after 57% profit                         │
│ increase](https://www.cbc.ca/news/business/scotiabank-hikes-dividend-after-57-profit-increase-1. │
│ 1132777)                                                                                         │
│   5. BMO Financial Group — 2013-08-27 — [BMO Financial Group Reports Third Quarter 2013          │
│ Results](https://newsroom.bmo.com/press-releases/bmo-financial-group-reports-third-quarter-2013- │
│ results-201308270889241001) (Information regarding dividend stability was found in the search    │
│ summary for this query                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [38]:
console.print("[dim]Grading with LLM judge against ground truth...[/dim]\n")

result = await evaluate_bloomberg_groundtruth_async(
    question=example.input,
    answer=eval_response.text,
    ground_truth=example.expected_output,
    answer_type=example.metadata.answer_type,
    model_config=LLMRequestConfig(temperature=0.0),
)

outcome_color = {
    EvaluationOutcome.FULLY_CORRECT: "green",
    EvaluationOutcome.CORRECT_WITH_EXTRANEOUS: "yellow",
    EvaluationOutcome.PARTIALLY_CORRECT: "orange1",
    EvaluationOutcome.FULLY_INCORRECT: "red",
}.get(result.outcome, "white")

metrics_table = Table(title="Grader Results")
metrics_table.add_column("Metric", style="cyan")
metrics_table.add_column("Value", style="white")
metrics_table.add_row("Outcome", f"[{outcome_color}]{result.outcome.value}[/{outcome_color}]")
metrics_table.add_row("Precision", f"{result.precision:.3f}")
metrics_table.add_row("Recall", f"{result.recall:.3f}")
metrics_table.add_row("F1", f"[bold]{result.f1_score:.3f}[/bold]")
metrics_table.add_row("Hallucination", f"{result.hallucination:.3f}")
metrics_table.add_row("Coherence", f"{result.coherence:.3f}")
metrics_table.add_row("Coverage", f"{result.coverage:.3f}")
console.print(metrics_table)

if result.explanation:
    console.print(Panel(result.explanation, title="Grader Explanation", border_style="magenta"))

# Show per-item correctness for Set Answer questions
if result.correctness_details:
    details_table = Table(title="Correctness Details")
    details_table.add_column("Expected Item", style="white")
    details_table.add_column("Found", style="cyan", justify="center")
    for item, found in result.correctness_details.items():
        icon = "[green]\u2713[/green]" if found else "[red]\u2717[/red]"
        label = item[:60] + "..." if len(item) > 60 else item
        details_table.add_row(label, icon)
    console.print(details_table)

Grading with LLM judge against ground truth...

              Grader Results               
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric        ┃ Value                   ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Outcome       │ correct_with_extraneous │
│ Precision     │ 0.600                   │
│ Recall        │ 1.000                   │
│ F1            │ 0.750                   │
│ Hallucination │ 0.000                   │
│ Coherence     │ 0.000                   │
│ Coverage      │ 0.000                   │
└───────────────┴─────────────────────────┘

╭─────────────────────────────────────── Grader Explanation ───────────────────────────────────────╮
│ The AI correctly identifies all three banks mentioned in the correct answer (RBC, TD, and        │
│ Scotiabank) that raised their dividends. The new dividend amounts and percentage increases are   │
│ all correct or within acceptable rounding limits based on the financial equivalence rules. For   │
│ Scotiabank, the AI provides the absolute increase ($0.02) instead of the percentage, which is    │
│ still a correct way to answer 'by how much'. The response also includes additional, correct      │
│ information about the two Big Five banks that did not raise dividends, which are considered      │
│ excessive answers as they were not in the ground truth.                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

                            Correctness Details                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Expected Item                                                   ┃ Found ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Royal Bank of Canada raised its dividend 6.3 percent to 67 c... │   ✓   │
│ Toronto-Dominion raised 4.9 percent to 85 cents                 │   ✓   │
│ Bank of Nova Scotia raised 3.3 percent to 62 cents              │   ✓   │
└─────────────────────────────────────────────────────────────────┴───────┘

## 3. Running the Evaluation Experiment

`run_experiment` runs the agent against every item in the Langfuse dataset, scores each
response, and records results in Langfuse. Each call creates a new named experiment run
that you can compare to previous runs in the UI.

The experiment takes two functions:

- **`agent_task`** — receives a dataset item, runs the agent, returns the answer string
- **`deepsearchqa_evaluator`** — receives question, answer, and ground truth; returns grader scores

> **Note:** This makes one agent call and one grader call per item. With 10 items and
> `max_concurrency=1`, expect 20–40 minutes depending on model latency.

In [28]:
async def agent_task(*, item: Any, **kwargs: Any) -> str:
    """Run the Knowledge Agent on a Langfuse dataset item."""
    agent = KnowledgeGroundedAgent(enable_planning=True)
    response = await agent.answer_async(item.input)
    return response.text


async def bloomberg_groundtruth_evaluator(
    *,
    input: str,  # noqa: A002
    output: str,
    expected_output: str,
    metadata: dict[str, Any] | None = None,
    **kwargs: Any,
) -> list[Evaluation]:
    """LLM-as-judge grader comparing agent answer to ground truth."""
    answer_type = (metadata or {}).get("answer_type", "Single Answer")
    result = await evaluate_bloomberg_groundtruth_async(
        question=input,
        answer=output,
        ground_truth=expected_output,
        answer_type=answer_type,
        model_config=LLMRequestConfig(temperature=0.0),
    )
    return result.to_evaluations()

In [29]:
experiment_result = run_experiment(
    DATASET_NAME,
    name="knowledge-agent-baseline",
    task=agent_task,
    evaluators=[bloomberg_groundtruth_evaluator],
    description="Baseline Knowledge Agent on August 2013 News questions.",
    max_concurrency=5,
)

console.print("[green]✓[/green] Experiment complete")
if experiment_result.dataset_run_url:
    display(
        HTML(
            f'<p>View experiment: <a href="{experiment_result.dataset_run_url}" target="_blank">{experiment_result.dataset_run_url}</a></p>'
        )
    )

2026-04-19 21:39:11,174 WARNING langfuse: Propagated attribute 'experiment_item_metadata' value is over 200 characters (231 chars). Dropping value.
2026-04-19 21:39:11,380 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: What was the Bank of Canada's policy interest rate in August 2013 and was it expected to change?...
2026-04-19 21:39:11,503 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-04-19 21:39:11,510 WARNING langfuse: Propagated attribute 'experiment_item_metadata' value is over 200 characters (214 chars). Dropping value.
2026-04-19 21:39:11,683 INFO aieng.agent_evals.knowledge_qa.agent: Answering question: What was the share price performance of Big Five Canadian bank stocks relative to the S&P/TSX Financ...
2026-04-19 21:39:11,819 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stre

✓ Experiment complete

## 4. Inspecting Results

The `ExperimentResult` object gives programmatic access to every item-level score.
Aggregate metrics are visible in the Langfuse experiment run summary in the UI.

In [30]:
rows = []
for item_result in experiment_result.item_results:
    item = item_result.item
    question = str(item.input)
    row = {
        "question": question[:55] + "..." if len(question) > 55 else question,
        "answer_type": (item.metadata or {}).get("answer_type", ""),
    }
    for evaluation in item_result.evaluations or []:
        row[evaluation.name] = evaluation.value
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

                                                  question   answer_type                 Outcome       F1  Precision   Recall  Hallucination  Coherence  Coverage
What was the Bank of Canada's policy interest rate in A... Single Answer         Fully Incorrect 0.000000   0.000000 0.000000            0.0        0.0       0.0
What was Canada's second-quarter GDP growth rate in 201... Single Answer         Fully Incorrect 0.000000   0.000000 0.000000            0.0        0.0       0.0
By how much did RBC and Scotiabank each raise their div... Single Answer           Fully Correct 1.000000   1.000000 1.000000            0.0        0.0       0.0
What was the share price performance of Big Five Canadi... Single Answer         Fully Incorrect 0.000000   0.000000 0.000000            0.0        0.0       0.0
Did any of the Big Five Canadian banks announce layoffs... Single Answer         Fully Incorrect 0.000000   0.000000 0.000000            0.0        0.0       0.0
What was BMO's provision for

In [31]:
# Mean of numeric metrics
numeric_cols = [c for c in ["F1", "Precision", "Recall", "Hallucination", "Coherence", "Coverage"] if c in df.columns]
if numeric_cols:
    means_table = Table(title="Mean Scores")
    means_table.add_column("Metric", style="cyan")
    means_table.add_column("Mean", style="white")
    for col in numeric_cols:
        means_table.add_row(col, f"{df[col].mean():.3f}")
    console.print(means_table)

# Outcome distribution
if "Outcome" in df.columns:
    outcome_table = Table(title="Outcome Distribution")
    outcome_table.add_column("Outcome", style="cyan")
    outcome_table.add_column("Count", style="white", justify="right")
    outcome_table.add_column("Fraction", style="dim", justify="right")
    total = len(df)
    for outcome, count in df["Outcome"].value_counts().items():
        outcome_table.add_row(str(outcome), str(count), f"{count / total:.0%}")
    console.print(outcome_table)

       Mean Scores       
┏━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric        ┃ Mean  ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━┩
│ F1            │ 0.416 │
│ Precision     │ 0.403 │
│ Recall        │ 0.464 │
│ Hallucination │ 0.000 │
│ Coherence     │ 0.000 │
│ Coverage      │ 0.000 │
└───────────────┴───────┘

             Outcome Distribution             
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ Outcome                 ┃ Count ┃ Fraction ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ Fully Incorrect         │    15 │      50% │
│ Fully Correct           │    10 │      33% │
│ Correct with Extraneous │     3 │      10% │
│ Partially Correct       │     2 │       7% │
└─────────────────────────┴───────┴──────────┘

## 5. Iterating on the Agent

The dataset in Langfuse is persistent — you don't need to re-upload it. To evaluate a modified
agent, call `run_experiment` again with a new `name` argument. Langfuse will create a new
experiment run and you can compare runs side-by-side in the UI.

### Levers to Explore

- **System prompt** — edit `SYSTEM_INSTRUCTIONS_TEMPLATE` in `system_instructions.py` to change
  the search strategy, verification rules, or final answer format
- **Planning** — toggle `enable_planning=False` to skip PlanReAct and compare quality vs. speed
- **Model** — change the Gemini model in `KnowledgeGroundedAgent` for different capability/cost trade-offs
- **Dataset** — change the `category` filter in Section 1 or increase `samples` to cover more examples

### What to Look for in Langfuse

- Items with **low F1** — did the agent fail to fetch the source? Stop early? Misread the question?
- Items with **`correct_with_extraneous`** — is the agent over-generating? Can the prompt be tightened?
- **Latency outliers** — which steps are slow? Is replanning happening unnecessarily?

## Summary

In this notebook you:

1. **Uploaded** a DeepSearchQA subset to Langfuse as a persistent, reusable dataset
2. **Understood** the LLM-as-judge grader: precision, recall, F1, and the four outcome categories
3. **Walked through** a single-sample evaluation end-to-end
4. **Ran** a full experiment with `run_experiment` and inspected item-level scores
5. **Learned** how to iterate: re-run with a new experiment name to compare configurations in Langfuse

The evaluation pipeline is the foundation for systematic agent improvement — each iteration
produces a new experiment run that you can compare to the baseline in the Langfuse UI.

In [20]:
console.print(Panel("[green]✓[/green] Notebook complete!", title="Done", border_style="green"))
if experiment_result.dataset_run_url:
    display(
        HTML(
            f'<p>View experiment results: <a href="{experiment_result.dataset_run_url}" target="_blank">{experiment_result.dataset_run_url}</a></p>'
        )
    )

╭────────────────────────────────────────────── Done ──────────────────────────────────────────────╮
│ ✓ Notebook complete!                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯